## Import required libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
#catalog = "shopvista"

In [0]:
dbutils.widgets.text('catalog', 'abc','catalog')
dbutils.widgets.text('data_source','xyz', 'data_source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')
print(catalog, data_source)

In [0]:
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

## Data Cleaning and Transformation

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("brand_code",F.regexp_replace(F.col("brand_code"), r"[^a-zA-Z0-9]", ""))
display(df_silver.limit(10))

In [0]:
df_silver.select("category_code").distinct()

df_silver.show(10)

In [0]:
# Replace the incorrect category codes with the right one
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])

# ✅ Show results
df_silver.select("category_code").distinct().show()

In [0]:
# write to silver layer
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .option('mergeschema', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
